<a href="https://colab.research.google.com/github/VankaVijay84/ML/blob/main/Project/Box_Office_Revenue_Prediction_RandomForest_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Box Office Revenue Prediction using Random Forest and XGBoost

**Machine Learning Regression Project**

This Google Colab notebook predicts a movie's worldwide box-office revenue using:
1. Random Forest Regressor
2. XGBoost Regressor

The workflow includes data loading, EDA, preprocessing, feature engineering, model training, evaluation, model comparison, feature importance, and prediction for a new movie.

**Target:** `Box_Office_Revenue_Million_USD`


## 1. Install and import libraries

In [ ]:
!pip -q install xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


## 2. Upload and load the dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
df = pd.read_csv("box_office_revenue_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()


## 3. Understand the dataset

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nStatistical summary:")
display(df.describe(include="all").T)


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df["Box_Office_Revenue_Million_USD"], kde=True)
plt.title("Distribution of Box Office Revenue")
plt.xlabel("Revenue (Million USD)")
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x="Budget_Million_USD", y="Box_Office_Revenue_Million_USD", alpha=0.6)
plt.title("Budget vs Box Office Revenue")
plt.show()


In [ ]:
plt.figure(figsize=(12,6))
genre_revenue = df.groupby("Genre")["Box_Office_Revenue_Million_USD"].mean().sort_values(ascending=False)
sns.barplot(x=genre_revenue.values, y=genre_revenue.index)
plt.title("Average Revenue by Genre")
plt.xlabel("Average Revenue (Million USD)")
plt.ylabel("Genre")
plt.show()


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
plt.figure(figsize=(13,9))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


## 5. Prepare features and target

In [ ]:
target = "Box_Office_Revenue_Million_USD"

X = df.drop(columns=[target, "Movie_ID", "Movie_Title"])
y = df[target]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)


## 6. Train-test split and preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 7. Random Forest Regressor

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Results")
print(f"MAE  : {rf_mae:.2f} million USD")
print(f"RMSE : {rf_rmse:.2f} million USD")
print(f"R²   : {rf_r2:.4f}")


## 8. XGBoost Regressor

In [ ]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost Results")
print(f"MAE  : {xgb_mae:.2f} million USD")
print(f"RMSE : {xgb_rmse:.2f} million USD")
print(f"R²   : {xgb_r2:.4f}")


## 9. Compare Random Forest and XGBoost

In [ ]:
results = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "MAE": [rf_mae, xgb_mae],
    "RMSE": [rf_rmse, xgb_rmse],
    "R2": [rf_r2, xgb_r2]
})

display(results.sort_values("R2", ascending=False))

best_model_name = results.loc[results["R2"].idxmax(), "Model"]
print("Best model based on R²:", best_model_name)


## 10. Actual vs Predicted Revenue

In [ ]:
plt.figure(figsize=(9,7))
plt.scatter(y_test, rf_pred, alpha=0.55, label="Random Forest")
plt.scatter(y_test, xgb_pred, alpha=0.55, label="XGBoost")

min_val = min(y_test.min(), rf_pred.min(), xgb_pred.min())
max_val = max(y_test.max(), rf_pred.max(), xgb_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.xlabel("Actual Revenue (Million USD)")
plt.ylabel("Predicted Revenue (Million USD)")
plt.title("Actual vs Predicted Box Office Revenue")
plt.legend()
plt.show()


## 11. Feature importance

In [ ]:
# Extract transformed feature names
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

rf_importance = rf_model.named_steps["model"].feature_importances_
rf_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_importance
}).sort_values("Importance", ascending=False).head(20)

plt.figure(figsize=(10,8))
sns.barplot(data=rf_importance_df, x="Importance", y="Feature")
plt.title("Top Random Forest Feature Importances")
plt.show()

display(rf_importance_df)


In [ ]:
xgb_importance = xgb_model.named_steps["model"].feature_importances_
xgb_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_importance
}).sort_values("Importance", ascending=False).head(20)

plt.figure(figsize=(10,8))
sns.barplot(data=xgb_importance_df, x="Importance", y="Feature")
plt.title("Top XGBoost Feature Importances")
plt.show()

display(xgb_importance_df)


## 12. Predict revenue for a new movie

In [ ]:
new_movie = pd.DataFrame([{
    "Release_Year": 2026,
    "Genre": "Action",
    "Language": "English",
    "Certification": "PG-13",
    "Production_Company": "Alpha Studios",
    "Budget_Million_USD": 100,
    "Marketing_Million_USD": 30,
    "Runtime_Minutes": 135,
    "IMDb_Rating": 7.5,
    "IMDb_Votes": 80000,
    "Theater_Count": 3500,
    "Opening_Weekend_Million_USD": 120,
    "Is_Sequel": 1,
    "Is_Franchise": 1
}])

rf_new_prediction = rf_model.predict(new_movie)[0]
xgb_new_prediction = xgb_model.predict(new_movie)[0]

print(f"Random Forest predicted revenue: ${rf_new_prediction:.2f} million")
print(f"XGBoost predicted revenue: ${xgb_new_prediction:.2f} million")


## 13. Save the trained models

In [ ]:
import joblib

joblib.dump(rf_model, "random_forest_box_office_model.pkl")
joblib.dump(xgb_model, "xgboost_box_office_model.pkl")

print("Models saved successfully.")


## 14. Download models

In [ ]:
from google.colab import files

files.download("random_forest_box_office_model.pkl")
files.download("xgboost_box_office_model.pkl")


## 15. Project conclusion

This project demonstrates a complete supervised machine-learning regression workflow.

**Random Forest** builds many decision trees and averages their predictions.

**XGBoost** builds boosted decision trees sequentially, where later trees focus on correcting previous errors.

The models are compared using:
- **MAE:** average absolute prediction error.
- **RMSE:** penalizes larger errors more strongly.
- **R²:** proportion of target variance explained by the model.

For a production system, the next improvements would be hyperparameter tuning, cross-validation, richer real-world movie metadata, inflation adjustment, and external validation on a genuinely unseen dataset.
